# Clase 18/08 — Funciones, dimensiones y la sigmoide
**Natural Language Processing — UFM**

Dos ideas que van juntas: cómo escala una función lineal cuando le agregamos parámetros, y cómo la
sigmoide convierte la salida de esa función en una probabilidad.

In [29]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110

## 1. El orden de las funciones según cuántos parámetros tienen

Siempre es la **misma fórmula**. Lo único que cambia es en cuántas dimensiones vive.

$$f(\mathbf{x}) = w_1x_1 + w_2x_2 + \dots + w_nx_n + b = \mathbf{w}\cdot\mathbf{x} + b$$

| features (n) | parámetros | la función es | vive en | la frontera es |
|---|---|---|---|---|
| 0 | 1 → $b$ | $f = b$ | recta horizontal en $\mathbb{R}^2$ | un punto |
| 1 | 2 → $w, b$ | $f(x) = wx + b$ | una **recta** en $\mathbb{R}^2$ | un punto |
| 2 | 3 → $w_1, w_2, b$ | $f(x_1,x_2) = w_1x_1 + w_2x_2 + b$ | un **plano** en $\mathbb{R}^3$ | una recta |
| 3 | 4 | $\mathbf{w}\cdot\mathbf{x} + b$ | un **hiperplano** en $\mathbb{R}^4$ | un plano |
| n | n+1 | $\mathbf{w}\cdot\mathbf{x} + b$ | un **hiperplano** en $\mathbb{R}^{n+1}$ | un hiperplano de dimensión n−1 |

Las dos reglas que hay que retener:

1. **Parámetros = n + 1.** Un peso por feature, más el sesgo $b$.
2. **La frontera de decisión siempre tiene una dimensión menos que el espacio de entrada.** En 1D
   separás con un punto, en 2D con una recta, en 3D con un plano, en nD con un hiperplano.

A partir de 3 features ya no se puede dibujar, pero el álgebra no cambia en absoluto: sigue siendo
un producto punto más una constante.

In [30]:
x = np.linspace(-4, 4, 200)

fig, ax = plt.subplots(1, 3, figsize=(14, 4))

# 0 features -> 1 parametro: una constante
ax[0].axhline(1.5, color="steelblue", lw=2)
ax[0].set_title("1 parametro:  f = b\n(constante)")
ax[0].set_xlabel("x"); ax[0].set_ylabel("f")

# 1 feature -> 2 parametros: una recta
for w, b, c in [(1.0, 0.5, "steelblue"), (-0.7, 1.0, "indianred"), (2.0, -1.0, "darkseagreen")]:
    ax[1].plot(x, w * x + b, color=c, lw=2, label=f"w={w}, b={b}")
ax[1].set_title("2 parametros:  f(x) = wx + b\n(recta)")
ax[1].set_xlabel("x"); ax[1].legend(fontsize=8)

# 1 feature, frontera = un punto (donde f cruza 0)
w, b = 1.5, -2.0
ax[2].plot(x, w * x + b, color="steelblue", lw=2)
ax[2].axhline(0, color="gray", ls="--", lw=1)
ax[2].axvline(-b / w, color="crimson", ls="--", lw=1.5)
ax[2].scatter([-b / w], [0], color="crimson", zorder=5, s=60)
ax[2].set_title(f"La frontera es un PUNTO\nx = {-b/w:.2f}  (donde f = 0)")
ax[2].set_xlabel("x")

for a in ax:
    a.grid(alpha=.3)
plt.tight_layout()
plt.show()

<Figure size 1540x440 with 3 Axes>

In [31]:
# 2 features -> 3 parametros: un PLANO en 3D, y su frontera es una RECTA en el plano x1-x2
w1, w2, b = 1.0, -1.5, 0.5

g = np.linspace(-3, 3, 40)
X1, X2 = np.meshgrid(g, g)
F = w1 * X1 + w2 * X2 + b

fig = plt.figure(figsize=(13, 5))

ax1 = fig.add_subplot(1, 2, 1, projection="3d")
ax1.plot_surface(X1, X2, F, cmap="coolwarm", alpha=.85, linewidth=0)
ax1.contour(X1, X2, F, levels=[0], colors="black", linewidths=3, offset=F.min())
ax1.set_xlabel("x1"); ax1.set_ylabel("x2"); ax1.set_zlabel("f")
ax1.set_title("3 parametros: f(x1,x2) = w1*x1 + w2*x2 + b\nes un PLANO en R3")

ax2 = fig.add_subplot(1, 2, 2)
cs = ax2.contourf(X1, X2, F, levels=20, cmap="coolwarm")
ax2.contour(X1, X2, F, levels=[0], colors="black", linewidths=3)
plt.colorbar(cs, ax=ax2, label="f(x1,x2)")
ax2.set_xlabel("x1"); ax2.set_ylabel("x2")
ax2.set_title("Visto desde arriba: la frontera f = 0\nes una RECTA (una dimension menos)")

plt.tight_layout()
plt.show()

<Figure size 1430x550 with 3 Axes>

### Cómo se conecta con el corpus de noticias

En el clasificador del Laboratorio #3 el vocabulario tenía **16,131 palabras**. Eso significa
n = 16,131 features, o sea **16,132 parámetros por categoría** (un peso por palabra más el sesgo).
El hiperplano que separa las clases vive en un espacio de más de dieciséis mil dimensiones y es
imposible de dibujar, pero se calcula con la misma línea de álgebra: $\mathbf{w}\cdot\mathbf{x} + b$.

Por eso importa la dimensionalidad. Con 795 documentos de entrenamiento y 16,131 pesos por estimar
hay muchísimos más parámetros que ejemplos, que es exactamente la razón por la que recortar el
vocabulario a 1000 o 5000 palabras mejoraba el desempeño.

## 2. La función sigmoide

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

El problema que resuelve: $\mathbf{w}\cdot\mathbf{x} + b$ puede dar cualquier número real, de
$-\infty$ a $+\infty$. Una probabilidad tiene que estar entre 0 y 1. La sigmoide es el puente.

Propiedades:

- **Rango (0, 1)**, nunca llega exactamente a los extremos.
- **$\sigma(0) = 0.5$** — el punto de indecisión, justo sobre la frontera.
- **Simétrica:** $\sigma(-z) = 1 - \sigma(z)$.
- **Derivada:** $\sigma'(z) = \sigma(z)\,(1 - \sigma(z))$, con máximo de 0.25 en $z = 0$.
- **Satura** en los extremos: para $|z|$ grande la derivada se va a cero y el modelo casi deja de
  aprender. Es el problema del gradiente que desaparece.

In [32]:
def sigmoide(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-8, 8, 400)
s = sigmoide(z)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

ax[0].plot(z, s, color="steelblue", lw=2.5)
ax[0].axhline(0.5, color="gray", ls="--", lw=1)
ax[0].axvline(0, color="gray", ls="--", lw=1)
ax[0].axhline(0, color="lightgray", lw=1)
ax[0].axhline(1, color="lightgray", lw=1)
ax[0].scatter([0], [0.5], color="crimson", zorder=5, s=60)
ax[0].annotate("sigma(0) = 0.5", (0, 0.5), textcoords="offset points",
               xytext=(15, -25), color="crimson")
ax[0].set_title("Sigmoide")
ax[0].set_xlabel("z = w·x + b"); ax[0].set_ylabel("sigma(z)")

ax[1].plot(z, s * (1 - s), color="indianred", lw=2.5)
ax[1].axvline(0, color="gray", ls="--", lw=1)
ax[1].set_title("Derivada:  sigma(z)(1 - sigma(z))\nmaximo 0.25 en z = 0, satura en los extremos")
ax[1].set_xlabel("z")

for a in ax:
    a.grid(alpha=.3)
plt.tight_layout()
plt.show()

for valor in [-6, -2, -1, 0, 1, 2, 6]:
    print(f"sigma({valor:>3}) = {sigmoide(valor):.4f}")

<Figure size 1430x495 with 2 Axes>

sigma( -6) = 0.0025
sigma( -2) = 0.1192
sigma( -1) = 0.2689
sigma(  0) = 0.5000
sigma(  1) = 0.7311
sigma(  2) = 0.8808
sigma(  6) = 0.9975


In [33]:
# Que hacen los parametros: w controla la PENDIENTE, b DESPLAZA la curva
x = np.linspace(-8, 8, 400)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

for w in [0.3, 1, 3, 10]:
    ax[0].plot(x, sigmoide(w * x), lw=2, label=f"w = {w}")
ax[0].set_title("Efecto de w: que tan brusca es la transicion\n(w grande -> casi un escalon)")
ax[0].legend()

for b in [-4, -1, 0, 2]:
    ax[1].plot(x, sigmoide(x + b), lw=2, label=f"b = {b}")
ax[1].set_title("Efecto de b: donde queda la frontera\n(la curva se corre, no cambia de forma)")
ax[1].legend()

for a in ax:
    a.axhline(0.5, color="gray", ls="--", lw=1)
    a.set_xlabel("x"); a.grid(alpha=.3)
plt.tight_layout()
plt.show()

<Figure size 1430x495 with 2 Axes>

## 3. Las dos partes juntas

$$p = \sigma(\mathbf{w}\cdot\mathbf{x} + b)$$

La parte 1 pone el hiperplano; la parte 2 lo convierte en probabilidad. Y las dos cosas encajan en
un solo punto:

$$p = 0.5 \iff \sigma(z) = 0.5 \iff z = 0 \iff \mathbf{w}\cdot\mathbf{x} + b = 0$$

Es decir que **la frontera de decisión de la sigmoide es exactamente el hiperplano** de la primera
parte. La sigmoide no cambia dónde está la frontera, solo agrega una medida de confianza: qué tan
lejos del hiperplano cae el punto. Cerca de la frontera la probabilidad ronda 0.5 y el modelo duda;
lejos se acerca a 0 o a 1.

In [34]:
# Frontera de decision en 2D: el hiperplano de la parte 1 visto a traves de la sigmoide
w1, w2, b = 1.0, -1.5, 0.5

g = np.linspace(-4, 4, 300)
X1, X2 = np.meshgrid(g, g)
P = sigmoide(w1 * X1 + w2 * X2 + b)

fig, ax = plt.subplots(figsize=(7, 5.5))
cs = ax.contourf(X1, X2, P, levels=20, cmap="RdBu_r")
ax.contour(X1, X2, P, levels=[0.5], colors="black", linewidths=3)
plt.colorbar(cs, ax=ax, label="p = sigma(w·x + b)")
ax.set_xlabel("x1"); ax.set_ylabel("x2")
ax.set_title("La linea negra es p = 0.5,\nla misma recta w1*x1 + w2*x2 + b = 0")
plt.tight_layout()
plt.show()

<Figure size 770x605 with 2 Axes>

---

## 4. Naive Bayes vs Regresión logística

Las dos terminan trazando **el mismo tipo de frontera**: un hiperplano, exactamente el de la
sección 1. La diferencia no está en la forma de la frontera sino en **cómo llegan a ella**.

| | Naive Bayes | Regresión logística |
|---|---|---|
| Qué modela | $P(\mathbf{x} \mid c)\,P(c)$ — cómo se **generan** los datos | $P(c \mid \mathbf{x})$ — la frontera, directo |
| Tipo | **Generativo** | **Discriminativo** |
| Cómo aprende | contando frecuencias, de una pasada | optimizando (descenso de gradiente), iterativo |
| Supuesto fuerte | independencia condicional entre features | ninguno sobre la distribución de $\mathbf{x}$ |
| Features correlacionadas | cuenta la misma evidencia varias veces | reparte el peso entre ellas |
| Con pocos datos | **converge rápido**, sesgo alto pero varianza baja | necesita más ejemplos |
| Con muchos datos | se estanca (el supuesto lo limita) | **termina ganando** |
| Probabilidades | mal calibradas, muy pegadas a 0 y 1 | bien calibradas |
| Costo | entrenamiento casi instantáneo | más caro, hay que iterar |

La frase corta: **Naive Bayes aprende cómo es cada clase; la regresión logística aprende qué las
separa.**

In [35]:
from sklearn.datasets import make_classification
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

Xd, yd = make_classification(n_samples=400, n_features=2, n_redundant=0,
                             n_informative=2, n_clusters_per_class=1,
                             class_sep=1.2, random_state=7)

nb_g = GaussianNB().fit(Xd, yd)
lr   = LogisticRegression().fit(Xd, yd)

g1 = np.linspace(Xd[:, 0].min() - 1, Xd[:, 0].max() + 1, 300)
g2 = np.linspace(Xd[:, 1].min() - 1, Xd[:, 1].max() + 1, 300)
G1, G2 = np.meshgrid(g1, g2)
rejilla = np.c_[G1.ravel(), G2.ravel()]

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for a, (nombre, modelo) in zip(ax, [("Naive Bayes (gaussiano)", nb_g),
                                    ("Regresion logistica", lr)]):
    P = modelo.predict_proba(rejilla)[:, 1].reshape(G1.shape)
    cs = a.contourf(G1, G2, P, levels=20, cmap="RdBu_r", alpha=.8)
    a.contour(G1, G2, P, levels=[0.5], colors="black", linewidths=2.5)
    a.scatter(Xd[:, 0], Xd[:, 1], c=yd, cmap="RdBu_r", edgecolor="k", s=25)
    a.set_title(f"{nombre}\naccuracy = {accuracy_score(yd, modelo.predict(Xd)):.3f}")
    a.set_xlabel("x1"); a.set_ylabel("x2")
plt.colorbar(cs, ax=ax, label="P(clase 1)")
plt.show()

<Figure size 1430x550 with 3 Axes>

### 4.1 Dónde se rompe Naive Bayes: features correlacionadas

Duplicamos una feature informativa varias veces. La información **no aumenta**: es la misma columna
repetida. Pero Naive Bayes la multiplica en la verosimilitud como si fueran evidencias
independientes, y se vuelve cada vez más confiado en algo que ya sabía.

In [36]:
import pandas as pd

Xb, yb = make_classification(n_samples=600, n_features=6, n_informative=4,
                             n_redundant=0, class_sep=1.0, random_state=3)
Xtr, Xte, ytr, yte = train_test_split(Xb, yb, test_size=0.3, random_state=3, stratify=yb)

filas = []
for k in [0, 1, 3, 7, 15, 31]:
    # repetimos la columna 0 k veces mas (k=0 -> datos originales, sin duplicar nada)
    Xtr_k = np.hstack([Xtr] + [Xtr[:, [0]]] * k)
    Xte_k = np.hstack([Xte] + [Xte[:, [0]]] * k)

    nb_k = GaussianNB().fit(Xtr_k, ytr)
    lr_k = LogisticRegression(max_iter=2000).fit(Xtr_k, ytr)

    filas.append({
        "copias extra": k,
        "features": Xtr_k.shape[1],
        "Naive Bayes": round(accuracy_score(yte, nb_k.predict(Xte_k)), 4),
        "Reg. logistica": round(accuracy_score(yte, lr_k.predict(Xte_k)), 4),
        "confianza media NB": round(nb_k.predict_proba(Xte_k).max(axis=1).mean(), 4),
    })

tabla_dup = pd.DataFrame(filas).set_index("copias extra")
tabla_dup

,features,Naive Bayes,Reg. logistica,confianza media NB
copias extra,,,,
0,6,0.8333,0.8056,0.7982
1,7,0.8278,0.8056,0.8179
3,9,0.7667,0.8056,0.8586
7,13,0.7056,0.8056,0.9174
15,21,0.6722,0.8056,0.9564
31,37,0.6333,0.8056,0.9800


### 4.2 Cuántos datos necesita cada uno

Naive Bayes tiene **sesgo alto pero varianza baja**: su supuesto es falso, así que nunca llega a
ser óptimo, pero como solo cuenta frecuencias no necesita muchos ejemplos para estimar bien lo que
estima. La regresión logística no carga ese sesgo, pero tiene que ajustar un peso por feature y con
pocos datos se sobreajusta.

Lo que dice la teoría es que Naive Bayes converge más rápido y la regresión logística termina
ganando cuando hay suficientes datos. En la práctica el cruce no siempre se ve limpio: depende de
cuánto se viole el supuesto de independencia y de cuántas features haya frente a ejemplos. Conviene
mirar la curva de abajo y leer lo que realmente pasó, no lo que se esperaba.

In [37]:
Xc, yc = make_classification(n_samples=3000, n_features=20, n_informative=10,
                             n_redundant=5, class_sep=0.9, random_state=11)
Xtr_c, Xte_c, ytr_c, yte_c = train_test_split(Xc, yc, test_size=0.3,
                                              random_state=11, stratify=yc)

tamanos = [15, 25, 50, 100, 200, 400, 800, 1600, 2100]
acc_nb, acc_lr = [], []

for n in tamanos:
    idx = np.arange(n)
    acc_nb.append(accuracy_score(yte_c, GaussianNB()
                                 .fit(Xtr_c[idx], ytr_c[idx]).predict(Xte_c)))
    acc_lr.append(accuracy_score(yte_c, LogisticRegression(max_iter=3000)
                                 .fit(Xtr_c[idx], ytr_c[idx]).predict(Xte_c)))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(tamanos, acc_nb, "o-", color="steelblue", lw=2, label="Naive Bayes")
ax.plot(tamanos, acc_lr, "o-", color="indianred", lw=2, label="Regresion logistica")
ax.set_xscale("log")
ax.set_xlabel("Documentos de entrenamiento (escala log)")
ax.set_ylabel("Accuracy en prueba")
ax.set_title("Quien gana depende de cuantos datos hay")
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout()
plt.show()

pd.DataFrame({"n entrenamiento": tamanos,
              "Naive Bayes": np.round(acc_nb, 4),
              "Reg. logistica": np.round(acc_lr, 4)}).set_index("n entrenamiento")

<Figure size 880x550 with 1 Axes>

,Naive Bayes,Reg. logistica
n entrenamiento,,
15,0.6111,0.6133
25,0.6378,0.6833
50,0.6644,0.6789
100,0.7056,0.6944
200,0.7100,0.7044
400,0.7289,0.7144
800,0.7322,0.7456
1600,0.7456,0.7422
2100,0.7478,0.7411


### 4.3 Resumen

Las dos trazan un hiperplano, pero llegan por caminos opuestos. Naive Bayes estima cómo se ve cada
clase por dentro y deduce la frontera; la regresión logística busca la frontera directamente y
nunca se pregunta cómo se generan los datos.

De ahí salen las diferencias prácticas. Naive Bayes entrena de una sola pasada y funciona con
pocos ejemplos, pero cuando las features están correlacionadas —y en texto lo están casi siempre,
porque las palabras aparecen en grupos— cuenta la misma evidencia varias veces y termina
sobreconfiado. La regresión logística reparte el peso entre features correlacionadas y da
probabilidades más creíbles, a cambio de necesitar más datos y más cómputo.

Para el corpus de noticias del laboratorio anterior, con 795 documentos de entrenamiento y más de
dieciséis mil features, Naive Bayes está en su terreno favorable: muchísimas más dimensiones que
ejemplos. La regresión logística probablemente lo alcance solo si se recorta el vocabulario o se
agrega regularización, que es justamente lo que se vio al limitar el vocabulario a 1000 o 5000
palabras.

---

## 5. Matriz de confusión y métricas

Toda evaluación binaria sale de contar cuatro casos:

| | **Predicho: positivo** | **Predicho: negativo** |
|---|---|---|
| **Real: positivo** | **VP** (verdadero positivo) | **FN** (falso negativo) |
| **Real: negativo** | **FP** (falso positivo) | **VN** (verdadero negativo) |

La diagonal son los aciertos. Los dos errores no son intercambiables: un **FP** es una falsa
alarma, un **FN** es algo que se dejó pasar.

### Las fórmulas

**Accuracy** — de todo lo que clasifiqué, cuánto acerté.

$$\text{Accuracy} = \frac{VP + VN}{VP + VN + FP + FN}$$

**Precision** — de todo lo que dije que era positivo, cuánto lo era de verdad.

$$\text{Precision} = \frac{VP}{VP + FP}$$

**Recall** — de todos los positivos que existían, cuántos encontré.

$$\text{Recall} = \frac{VP}{VP + FN}$$

**F1** — la media armónica de las dos. Se usa la armónica y no el promedio simple porque castiga
que una de las dos esté baja.

$$F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

### Cómo leerlas

Precision y recall se mueven en direcciones opuestas. Si el modelo se vuelve más exigente para
declarar un positivo, sube la precision y baja el recall; si se vuelve más permisivo, pasa lo
contrario. Por eso ninguna de las dos sirve sola y existe el F1.

El caso extremo se vio en el Laboratorio #3: con TF-IDF, Macroeconomía tenía recall 1.00 y
precision 0.43. Encontró todas las de Macroeconomía porque predijo Macroeconomía para casi todo.
Con la accuracy sola ese fracaso pasaba desapercibido.

In [38]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

# Usamos los dos modelos de la seccion 4 sobre los mismos datos
resultados = {}

for nombre, modelo in [("Naive Bayes", nb_g), ("Reg. logistica", lr)]:
    pred = modelo.predict(Xd)
    VN, FP, FN, VP = confusion_matrix(yd, pred).ravel()   # orden de sklearn en binario

    # Calculadas a mano con las formulas de arriba
    accuracy  = (VP + VN) / (VP + VN + FP + FN)
    precision = VP / (VP + FP)
    recall    = VP / (VP + FN)
    f1        = 2 * precision * recall / (precision + recall)

    resultados[nombre] = {
        "VP": VP, "FP": FP, "FN": FN, "VN": VN,
        "accuracy": round(accuracy, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "F1": round(f1, 4),
        "F1 sklearn": round(f1_score(yd, pred), 4),   # verificacion
    }

# Filas = metricas, columnas = modelos
pd.DataFrame(resultados)

,Naive Bayes,Reg. logistica
VP,191.0000,193.0000
FP,2.0000,2.0000
FN,7.0000,5.0000
VN,200.0000,200.0000
accuracy,0.9775,0.9825
precision,0.9896,0.9897
recall,0.9646,0.9747
F1,0.9770,0.9822
F1 sklearn,0.9770,0.9822


In [39]:
# Las dos matrices de confusion, lado a lado
import seaborn as sns

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))

for a, (nombre, modelo) in zip(ax, [("Naive Bayes", nb_g), ("Reg. logistica", lr)]):
    cm = confusion_matrix(yd, modelo.predict(Xd))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=a,
                xticklabels=["neg", "pos"], yticklabels=["neg", "pos"])
    a.set_title(f"Matriz de confusion — {nombre}")
    a.set_xlabel("Prediccion"); a.set_ylabel("Real")

plt.tight_layout()
plt.show()

<Figure size 1210x495 with 2 Axes>